In [1]:
!pip uninstall -y transformers awq autoawq
!pip install "transformers==4.56.2" "autoawq==0.2.9"


import torch

if not torch.cuda.is_available():
    raise SystemError(
        "No GPU found. In Colab, go to Runtime → Change runtime type → Hardware accelerator → GPU."
    )

device = "cuda"
print("Using device:", device)

!pip install -q autoawq
!pip install transformers accelerate datasets psutil

import os, psutil, time, math
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"

def cpu_mem_mb():
    p = psutil.Process(os.getpid())
    return p.memory_info().rss / 1024**2

def gpu_mem_mb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1024**2

def describe_memory(label):
    print(f"[{label}] CPU: {cpu_mem_mb():7.2f} MB | GPU: {gpu_mem_mb():7.2f} MB")

@torch.no_grad()
def perplexity(model, tokenizer, text: str) -> float:
    model.eval()
    enc = tokenizer(text, return_tensors="pt").to(device)
    out = model(**enc, labels=enc["input_ids"])
    loss = out.loss.item()
    return math.exp(loss)

@torch.no_grad()
def timed_generate(model, tokenizer, prompt: str, max_new_tokens=40, runs=3):
    model.eval()
    times, last = [], None
    for _ in range(runs):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        torch.cuda.empty_cache()
        start = time.perf_counter()
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
        torch.cuda.synchronize()
        end = time.perf_counter()
        times.append(end - start)
        last = tokenizer.decode(out[0], skip_special_tokens=True)
    return sum(times) / len(times), last


Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 120.4 MB/s eta 0:00:00
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=b2aa6990334f8d03138b319dfa8c81fa01072b910dabfb84bfed75e2307d4c9c
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq
Using device: cuda


In [2]:
model_id = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

torch.cuda.empty_cache()
describe_memory("Before FP16 load")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device)

describe_memory("After FP16 load")
print("Baseline dtype:", next(model_fp16.parameters()).dtype)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

[After FP16 load] CPU: 1640.47 MB | GPU:  245.48 MB
Baseline dtype: torch.float16


In [3]:
calib_text = (
    "Quantization allows us to compress large language models like OPT while "
    "preserving most of their predictive power."
)
prompt = "In the future, efficient language models will"

ppl_fp16 = perplexity(model_fp16, tokenizer, calib_text)
t_fp16, out_fp16 = timed_generate(model_fp16, tokenizer, prompt)

print(f"Baseline FP16 perplexity: {ppl_fp16:.3f}")
print(f"Baseline FP16 gen time:   {t_fp16:.3f} s")
print("Baseline FP16 output:\n", out_fp16)


Baseline FP16 perplexity: 338.700
Baseline FP16 gen time:   1.040 s
Baseline FP16 output:
 In the future, efficient language models will be used to help us to understand the different types of languages.

The language model is a set of rules that are used to define the types of languages. The rules are used to define the


In [5]:


# quant_config = {
#     "zero_point": True,
#     "q_group_size": 128,
#     "w_bit": 4,
#     "version": "GEMM",   # GEMM kernels: good general default
# }

torch.cuda.empty_cache()
describe_memory("Before AWQ quantization")

start = time.perf_counter()

awq_model = AutoModelForCausalLM.from_pretrained(
    "ybelkada/opt-125m-awq",
    low_cpu_mem_usage=True,
    use_cache=False,
)

# # Activation-aware quantization: uses a small calibration set internally
# awq_model.quantize(
#     tokenizer,
#     quant_config=quant_config,
# )

# Move quantized model to GPU for inference
awq_model.to(device)

end = time.perf_counter()
describe_memory("After AWQ quantization")


[Before AWQ quantization] CPU: 2149.63 MB | GPU:  255.23 MB


config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

You have loaded an AWQ model on CPU and have a CUDA/XPU device available, make sure to set your model on a GPU device in order to run your model.


model.safetensors:   0%|          | 0.00/202M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[After AWQ quantization] CPU: 2186.04 MB | GPU:  374.68 MB


In [6]:
ppl_awq = perplexity(awq_model, tokenizer, calib_text)
t_awq, out_awq = timed_generate(awq_model, tokenizer, prompt)

print(f"AWQ 4-bit perplexity: {ppl_awq:.3f}")
print(f"AWQ 4-bit gen time:   {t_awq:.3f} s")
print("AWQ 4-bit output:\n", out_awq)

print("\nSummary:")
print(f"  FP16 perplexity: {ppl_fp16:.3f}")
print(f"  AWQ  perplexity: {ppl_awq:.3f}")
print(f"  FP16 gen time:   {t_fp16:.3f} s")
print(f"  AWQ  gen time:   {t_awq:.3f} s")


AWQ 4-bit perplexity: 306.888
AWQ 4-bit gen time:   2.159 s
AWQ 4-bit output:
 In the future, efficient language models will be used to provide a more efficient way to communicate with each other.

The following is a list of the most common language models used in the field of data science.

Data Science


Summary:
  FP16 perplexity: 338.700
  AWQ  perplexity: 306.888
  FP16 gen time:   1.040 s
  AWQ  gen time:   2.159 s


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
